# 🎬 אולפן המשנה — תמונות ניסיון ב־GPU חינמי

**מיועד לברכות א׳, א׳.** זו סביבת ניסיון פרטית ולא פרסום או אישור נוסח. אין שימוש בכלי התמונות או במכסות התמונות של ChatGPT.

1. ב־Colab: **Runtime → Change runtime type → GPU (למשל T4)**. השתמש במכסת ה־Free; אם יש לך מנוי Colab בתשלום, בדוק שלא נצרכים קרדיטים בתשלום.
2. לחץ **Runtime → Run all**. ברירת המחדל היא **שוט SH09 חדש**; לא חוזרים על ניסיון SH01 שכבר הושלם ונדחה. ניתן לבחור 5 תמונות של סצנה אחת, או 16 שוטים מתוכננים.
3. לאחר ההרצה יופיע קובץ ZIP להורדה. אין תוכנות להתקין במחשב שלך.

**אזהרה אמיתית:** GPU חינמי אינו מובטח. אם הוא לא זמין, המחברת עוצרת לפני הורדת מודל. משתמשים רק במשאבים שהשירות מעניק בחינם; אין קוד תשלום או API בתשלום.

**מקורות וזכויות:** המודל SDXL base שוחרר תחת CreativeML Open RAIL++-M; כל האיורים יהיו המחשה אמנותית ולא דיוקן או שחזור מאומת. אין לפרסם בלי ביקורת תורנית, בדיקת זכויות ואישור.


In [ ]:
# @title 🎞️ בחירת כמות התמונות (ללא שימוש ב-ChatGPT Image)
BATCH = "איור אחד חדש - SH09" # @param ["איור אחד חדש - SH09", "חמישה איורים - סצנת הבנים", "כל 16 איורי הניסיון"]
IMAGE_QUALITY = "איכות גבוהה - 1024x576" # @param ["איכות גבוהה - 1024x576", "מהיר - 768x448"]
AUTO_DOWNLOAD_ZIP = True # @param {type:"boolean"}
# הפעלה דרך מחברת Google Colab עם GPU חינמי בלבד; אין בקוד שירות תמונות בתשלום.


In [ ]:
import sys, platform
try:
    import torch
except ImportError as exc:
    raise RuntimeError("לא זוהתה סביבת Colab/PyTorch. פתח את המחברת ב-Google Colab.") from exc

if not torch.cuda.is_available():
    raise RuntimeError("אין GPU חינמי מחובר. ב-Colab בחר Runtime > Change runtime type > GPU ואז Run all. לא הופקה תמונה ולא נקרא שירות בתשלום.")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GIB = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
print("GPU:", GPU_NAME, "VRAM GiB:", GPU_GIB)
if GPU_GIB < 10:
    raise RuntimeError("ה-GPU קטן מדי למודל הניסיון. לא מבוצעת הורדת מודל או יצירת תמונות.")
print("זוהה GPU בחשבון. אי אפשר לקבוע מהקוד אם חשבונך משתמש במכסה חינמית או בקרדיטים; בדוק זאת לפני ההרצה.")


In [ ]:
import json, urllib.request, hashlib
BASE = "https://raw.githubusercontent.com/barkinam21-pixel/project-mishna/main/episodes/berakhot-01-01/"
PROMPT_SOURCE = BASE + "free-cloud-image-prompts.json"
with urllib.request.urlopen(PROMPT_SOURCE, timeout=30) as stream:
    plan = json.loads(stream.read().decode("utf-8"))
assert plan["episode_id"] == "berakhot-01-01"
assert plan["status"] == "private_style_tests_only"
assert plan["monetary_spend_authorized_usd"] == 0
assert plan["public_release_authorized"] is False
assert plan["torah_review_approved"] is False
assert plan["model_id"] == "stabilityai/stable-diffusion-xl-base-1.0"
assert len(plan["shots"]) == 16 and len({s["shot_id"] for s in plan["shots"]}) == 16
for s in plan["shots"]:
    assert s["shot_id"].startswith("SH") and s["generated"] is False and s["accepted"] is False
SELECTED_IDS = {
    "איור אחד חדש - SH09": ["SH09"],
    "חמישה איורים - סצנת הבנים": ["SH09", "SH10", "SH11", "SH12", "SH13"],
    "כל 16 איורי הניסיון": [s["shot_id"] for s in plan["shots"]],
}[BATCH]
selected = [s for s in plan["shots"] if s["shot_id"] in SELECTED_IDS]
assert len(selected) == len(SELECTED_IDS)
if IMAGE_QUALITY == "מהיר - 768x448":
    WIDTH, HEIGHT, STEPS = 768, 448, 16
else:
    WIDTH, HEIGHT, STEPS = 1024, 576, 24
print(f"נבחרו {len(selected)} איורי ניסיון, {WIDTH}x{HEIGHT}, {STEPS} צעדים. אין התקשרות לספק בתשלום.")


In [ ]:
import subprocess
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "diffusers>=0.35,<1", "transformers>=4.50,<5",
    "accelerate>=1.2,<2", "safetensors>=0.5"
])
print("ספריות חינמיות הותקנו על המכונה של Colab, לא במחשב האישי.")


In [ ]:
import os, time, zipfile, datetime, pathlib
from PIL import Image
from diffusers import StableDiffusionXLPipeline
from IPython.display import display, FileLink

OUT = pathlib.Path("/content/mishnah-free-image-trials")
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "prompt-pack-snapshot.json").write_text(json.dumps(plan, ensure_ascii=False, indent=2)+"\n", encoding="utf-8")
ZIP_PATH = pathlib.Path("/content/mishnah-free-image-trials.zip")
LEDGER = OUT / "usage-ledger.jsonl"

def cached_image_matches(path, shot_id, generation_spec_sha256):
    """Reuse only a byte-verified image with a matching complete generation specification."""
    if not path.is_file() or not LEDGER.is_file():
        return False
    actual_sha = hashlib.sha256(path.read_bytes()).hexdigest()
    for line in reversed(LEDGER.read_text(encoding="utf-8").splitlines()):
        try:
            record = json.loads(line)
        except ValueError:
            continue
        if record.get("shot_id") == shot_id and record.get("image_sha256") == actual_sha:
            return record.get("generation_spec_sha256") == generation_spec_sha256
    return False


pipe = StableDiffusionXLPipeline.from_pretrained(
    plan["model_id"],
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
)
pipe.enable_vae_slicing()
if GPU_GIB >= 14:
    pipe.to("cuda")
else:
    pipe.enable_model_cpu_offload()
pipe.set_progress_bar_config(disable=True)

def save_zip():
    tmp = ZIP_PATH.with_suffix(".zip.tmp")
    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for file in sorted(OUT.glob("*")):
            if file.is_file():
                zf.write(file, arcname=file.name)
    os.replace(tmp, ZIP_PATH)

successful = []
for index, shot in enumerate(selected, 1):
    image_path = OUT / (shot["shot_id"] + ".png")
    full_prompt = plan["style_lock"] + ". Shot direction: " + shot["shot_prompt"]
    prompt_digest = hashlib.sha256(full_prompt.encode("utf-8")).hexdigest()
    seed = int(shot["seed"])
    generation_spec = {
        "model_id": plan["model_id"], "model_revision": "unknown",
        "variant": "fp16", "prompt_sha256": prompt_digest,
        "negative_prompt_sha256": hashlib.sha256(plan["negative_prompt"].encode("utf-8")).hexdigest(),
        "seed": seed, "width": WIDTH, "height": HEIGHT,
        "inference_steps": STEPS, "guidance_scale": 6.0
    }
    spec_hash = hashlib.sha256(json.dumps(generation_spec, sort_keys=True).encode("utf-8")).hexdigest()
    # A same-named PNG alone is not evidence that its prompt/model/settings match.
    if cached_image_matches(image_path, shot["shot_id"], spec_hash):
        print("מטמון מאומת SHA+מפרט:", image_path.name)
        successful.append(image_path)
        continue
    if image_path.exists():
        old_hash = hashlib.sha256(image_path.read_bytes()).hexdigest()
        archived = OUT / (shot["shot_id"] + ".superseded-" + old_hash[:12] + ".png")
        if archived.exists() and hashlib.sha256(archived.read_bytes()).hexdigest() != old_hash:
            raise RuntimeError("Archive name collision; refusing to overwrite an old image")
        image_path.replace(archived)
        print("קובץ ישן נשמר בנפרד, לא ישמש בטעות כמטמון:", archived.name)
    started = time.monotonic()
    print(f"[{index}/{len(selected)}] יוצר {shot['shot_id']} על GPU של החשבון; ללא API בתשלום")
    try:
        generator = torch.Generator(device="cuda").manual_seed(seed)
        result = pipe(
            prompt=full_prompt,
            negative_prompt=plan["negative_prompt"],
            width=WIDTH,
            height=HEIGHT,
            num_inference_steps=STEPS,
            guidance_scale=6.0,
            generator=generator
        )
        image = result.images[0]
        image.save(image_path, format="PNG")
        sha = hashlib.sha256(image_path.read_bytes()).hexdigest()
        record = {
            "date_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "shot_id": shot["shot_id"], "scene_id": shot["scene_id"],
            "model_id": plan["model_id"], "model_revision": "unknown",
            "model_license": plan["model_license"],
            "seed": seed, "prompt_sha256": prompt_digest,
            "generation_spec_sha256": spec_hash, "generation_spec": generation_spec,
            "image_sha256": sha, "image_size": [WIDTH, HEIGHT],
            "inference_steps": STEPS, "gpu": GPU_NAME,
            "duration_seconds": round(time.monotonic()-started, 1),
            "operation": "image_generate", "accepted": False,
            "public_release_authorized": False,
            "paid_api_calls": 0, "external_cash_spend_initiated_usd": 0,
            "colab_gpu_quota_usage": "unknown",
            "chatgpt_image_quota_usage": 0
        }
        with LEDGER.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(record, ensure_ascii=False)+"\n")
        successful.append(image_path)
        save_zip()  # Preserve progress after each generated image.
        print("נשמר:", image_path.name, "| SHA-256:", sha[:12])
    except torch.cuda.OutOfMemoryError:
        print("GPU memory insufficient. Earlier images remain in ZIP if any.")
        torch.cuda.empty_cache()
        break
    except KeyboardInterrupt:
        print("עצירה ידנית; תמונות קודמות נשמרו ב-ZIP.")
        break
    except Exception as exc:
        print("עצרתי בלי לעקוף כשל או להפעיל שירות אחר:", type(exc).__name__, str(exc)[:150])
        break

if successful:
    save_zip()
    print(f"נוצרו {len(successful)} תמונות בסשן. כולן טיוטות לאישור אנושי.")
    display(Image.open(successful[0]).resize((640,360)))
    print("קישור ZIP להורדה:")
    display(FileLink(str(ZIP_PATH)))
    if AUTO_DOWNLOAD_ZIP:
        try:
            from google.colab import files
            files.download(str(ZIP_PATH))
        except ImportError:
            print("אפשר להוריד מקישור ה-ZIP שמוצג למעלה.")
else:
    print("לא נוצרה תמונה; אין שימוש ב-ChatGPT Images או שירות חיצוני בתשלום.")
